# Data Explorer: dataset overview and preprocessing

**Purpose:** Understand the raw dataset before scoring. Shows preprocessing pipeline (events → cleaned), coverage metrics, distribution across repos and event types, and sample comments.

**Working directory:** The first code cell adds the repo root to `sys.path`, so you can run with the kernel's cwd as either the repository root or the `notebooks/` folder. The database path is still `data/raw/events.db` under the repo.

In [1]:
import sys
from pathlib import Path

# Add repo root to sys.path so notebooks.lib can be imported
here = Path.cwd().resolve()
for p in [here, *here.parents]:
    if (p / "project_config.py").is_file():
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))
        break

In [2]:
from __future__ import annotations

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from notebooks.lib import connect, breakdown_repo_author_association, breakdown_event_type, breakdown_repo_event_type

%matplotlib inline

RANDOM_SEED = 42
CLEAN_SAMPLE_N = 12

## Preprocessing Pipeline

- **`events`**: raw GitHub events ingested by `dataset.py`
- **`cleaned`**: rows that survived preprocessing in `preprocess.py` (text extraction, bot filtering, text normalization, tokenization; see `preprocessing/workflow.py`)

Coverage metrics show how many rows move through each stage of the pipeline.

In [3]:
SQL_CLEANED_SAMPLE = """
SELECT
  c.id AS comment_id,
  json_extract(e.event_data, '$.type') AS event_type,
  json_extract(e.event_data, '$.repo.name') AS repo,
  date(json_extract(e.event_data, '$.created_at')) AS event_date,
  COALESCE(
    json_extract(e.event_data, '$.actor.login'),
    json_extract(e.event_data, '$.actor.display_login'),
    ''
  ) AS actor_login,
  substr(c.cleaned_text, 1, 200) AS cleaned_text_preview
FROM cleaned c
INNER JOIN events e ON e.id = c.id
"""

with connect() as conn:
    n_events = int(pd.read_sql("SELECT COUNT(*) AS n FROM events", conn)["n"].iloc[0])
    n_cleaned = int(pd.read_sql("SELECT COUNT(*) AS n FROM cleaned", conn)["n"].iloc[0])
    cleaned_sample = pd.read_sql(SQL_CLEANED_SAMPLE, conn)

pct_cleaned_vs_events = round(100.0 * n_cleaned / n_events, 2) if n_events else float("nan")

coverage_metrics = pd.DataFrame(
    [
        {"metric": "events (raw rows)", "count": n_events},
        {"metric": "cleaned (after preprocess)", "count": n_cleaned},
        {"metric": "coverage cleaned / events (%)", "coverage": pct_cleaned_vs_events},
    ]
)
coverage_metrics["count"] = coverage_metrics["count"].fillna(pd.NA).astype("Int64")
coverage_metrics["coverage"] = coverage_metrics["coverage"].astype("Float64")

display(Markdown("### Preprocessing Coverage"))
display(coverage_metrics)

### Preprocessing Coverage

,metric,count,coverage
0,events (raw rows),41936,<NA>
1,cleaned (after preprocess),37197,<NA>
2,coverage cleaned / events (%),<NA>,88.7


In [4]:
sample_cleaned = cleaned_sample.sample(
    n=min(CLEAN_SAMPLE_N, len(cleaned_sample)),
    random_state=RANDOM_SEED,
).sort_values(["repo", "event_date"])
sample_cleaned

,comment_id,event_type,repo,event_date,actor_login,cleaned_text_preview
70,34557567965,PullRequestReviewCommentEvent,django/django,2024-01-03,saippuakauppias,`exception` here is: ```python tests = [ broke...
8943,42319990559,PullRequestEvent,django/django,2024-09-27,saJaeHyukc,refs #35734 -- updated postgresql version in s...
33311,43898116171,PullRequestEvent,django/django,2024-11-17,Inzamamulhaq01,update shell.py auto import models #### trac t...
36476,44875693863,IssueCommentEvent,expressjs/express,2024-12-17,axhuwastaken,could you share more context about your projec...
21879,36094303605,PullRequestEvent,spring-projects/spring-boot,2024-02-28,philwebb,replace string concatenation with stringbuilde...
24288,39492466819,IssueCommentEvent,spring-projects/spring-boot,2024-06-21,mvitz,@philwebb thanks for your quick reply. anythin...
25951,41735962967,IssueCommentEvent,spring-projects/spring-boot,2024-09-09,philwebb,i'm not sure why `messagesourceproperties` is ...
31681,43434997196,PullRequestEvent,spring-projects/spring-boot,2024-11-01,philwebb,add date and uuid deserialization support in n...
37049,45105017303,IssueCommentEvent,spring-projects/spring-boot,2024-12-27,pivotal-cla,@taewoocode thank you for signing the [contrib...
29187,35846454429,IssueCommentEvent,tiangolo/fastapi,2024-02-20,nilslindemann,danke sehr!


## Distribution Across Repos and Event Types

Data broken down by repository and event type from the cleaned dataset.

In [5]:
display(Markdown("### Cleaned comments by repo"))
display(breakdown_repo_author_association("cleaned").groupby("repo")["n"].sum().sort_values(ascending=False))

display(Markdown("### Cleaned comments by event type"))
display(breakdown_event_type("cleaned"))

display(Markdown("### Cleaned comments by repo × event type"))
display(breakdown_repo_event_type("cleaned").head(20))

### Cleaned comments by repo

repo
django/django                  12565
spring-projects/spring-boot     8464
tiangolo/fastapi                4243
nestjs/nest                     3649
fastify/fastify                 3375
expressjs/express               3059
gin-gonic/gin                   1021
pallets/flask                    450
koajs/koa                        191
hapijs/hapi                      180
Name: n, dtype: int64

### Cleaned comments by event type

,event_type,n
0,IssueCommentEvent,19030
1,PullRequestEvent,7921
2,PullRequestReviewCommentEvent,7620
3,PullRequestReviewEvent,2626


### Cleaned comments by repo × event type

,repo,event_type,n
0,django/django,IssueCommentEvent,3334
1,django/django,PullRequestEvent,2358
2,django/django,PullRequestReviewCommentEvent,5092
3,django/django,PullRequestReviewEvent,1781
4,expressjs/express,IssueCommentEvent,1492
5,expressjs/express,PullRequestEvent,1146
6,expressjs/express,PullRequestReviewCommentEvent,278
7,expressjs/express,PullRequestReviewEvent,143
8,fastify/fastify,IssueCommentEvent,1602
9,fastify/fastify,PullRequestEvent,683


### Actor "role" in this dataset

- **`author_association`** (above) is GitHub's label for the author's relationship to the repo on that object (e.g. `MEMBER`, `OWNER`). It is not org job title.
- Richer roles (maintainer lists, teams) require the GitHub API or an external mapping merged by `login`.